# Antasena Super Course — Rekam Data Telemetri (MQTT) (KUNCI JAWABAN)

Notebook ini adalah subscriber MQTT: ia menerima data telemetri yang disiarkan instruktur, lalu menyimpannya ke CSV. Isinya sama dengan `subscriber.py`, hanya dipecah per sel supaya mudah dibaca.

**Cara kerja MQTT singkatnya:**
- **Broker** adalah server perantara (di sini `broker.hivemq.com`, port `1883`).
- **Publisher** (mobil, atau `publisher.py` milik instruktur) mengirim pesan ke sebuah **topik**, yaitu `antasena/course/telemetry`.
- **Subscriber** (kamu) mendaftar ke topik yang sama, lalu broker meneruskan setiap pesan baru ke semua subscriber.

Setiap pesan berisi satu baris data telemetri dalam format JSON.

## 1. Persiapan

Instal library MQTT `paho-mqtt`.

In [1]:
!pip install -q paho-mqtt

import csv, json, os, sys, time
import pandas as pd
import paho.mqtt.client as mqtt


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Terano\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Pengaturan

Alamat broker dan topik harus **sama persis** dengan yang dipakai publisher.

In [2]:
BROKER = "broker.hivemq.com"          # alamat broker
PORT = 1883                           # port MQTT
TOPIC = "antasena/course/telemetry"   # topik yang didengarkan

OUT_FILE = "rekaman_saya.csv"         # file hasil rekaman
DURASI_S = 60                         # berapa detik merekam

## 3. `on_connect`: saat terhubung ke broker

Fungsi ini dipanggil otomatis begitu broker menerima koneksi kita. Di sinilah kita *subscribe* ke topik.

In [3]:
def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code.is_failure:
        print(f"Gagal terhubung: {reason_code}")
        return
    print("Terhubung ke broker!")
    client.subscribe(TOPIC, qos=1)
    print(f"Subscribe ke topik: {TOPIC}")

## 4. `on_message`: saat ada pesan masuk

Fungsi ini dipanggil setiap kali ada pesan baru. Pesan JSON diubah menjadi satu baris CSV. Baris pertama file berisi nama kolom (header).

In [4]:
jumlah_pesan = 0


def on_message(client, userdata, msg):
    global jumlah_pesan
    payload = msg.payload.decode("utf-8")
    try:
        data = json.loads(payload)
    except json.JSONDecodeError:
        print("  (dilewati: bukan JSON)")
        return

    jumlah_pesan += 1
    print(f"Pesan #{jumlah_pesan}: {payload}")

    file_baru = not os.path.exists(OUT_FILE)
    with open(OUT_FILE, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(data.keys()))
        if file_baru:
            writer.writeheader()
        writer.writerow(data)

## 5. Terhubung dan mulai merekam

Jalankan sel ini **saat instruktur memberi aba-aba** bahwa `publisher.py` sudah berjalan.

Sel ini membuat client, memasang kedua fungsi di atas, terhubung ke broker, lalu terus memproses pesan masuk selama `DURASI_S` detik. Tekan tombol ■ (Stop) untuk berhenti lebih awal. File lama dihapus dulu agar rekaman dimulai dari awal.

Jika setelah *Subscribe ke topik* tidak ada pesan yang muncul, berarti instruktur belum menyiarkan data.

In [ ]:
if os.path.exists(OUT_FILE):
    os.remove(OUT_FILE)
jumlah_pesan = 0

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect(___, ___, keepalive=60)

selesai = time.time() + DURASI_S
try:
    while time.time() < selesai:
        client.loop(timeout=0.5)   # proses pesan masuk
except KeyboardInterrupt:
    print("Dihentikan.")
client.disconnect()
print(f"Selesai. {jumlah_pesan} pesan diterima.")

Terhubung ke broker!
Subscribe ke topik: antasena/course/telemetry
Dihentikan.
Selesai. 0 pesan diterima.


## 6. Cek hasil rekaman

In [ ]:
if os.path.exists(OUT_FILE):
    df = pd.read_csv(OUT_FILE)
    print(f"{len(df)} baris terekam")
    display(df.tail())
else:
    print("Belum ada data. Apakah instruktur sedang menjalankan publisher.py?")

## 7. Unduh CSV

File di Colab hilang saat runtime berakhir, jadi unduh CSV-mu jika ingin menyimpannya.

In [ ]:
if "google.colab" in sys.modules and os.path.exists(OUT_FILE):
    from google.colab import files
    files.download(OUT_FILE)

## 8. Bonus: menjalankan `subscriber.py` dari terminal

Sel-sel di atas adalah isi skrip `subscriber.py` yang dipecah per bagian. Skrip aslinya juga bisa dijalankan langsung sebagai file Python dari terminal. Sel di bawah mengunduh `subscriber.py` dari repo kursus.

In [ ]:
import os, urllib.request

URL = "https://raw.githubusercontent.com/Antasena-ITS-Team/ASC_Simulator/master/subscriber.py"
urllib.request.urlretrieve(URL, "subscriber.py")
print("subscriber.py tersimpan di", os.path.abspath("subscriber.py"))

Lalu buka terminal Colab dengan tombol **Terminal** di kiri bawah, dan jalankan:

```
cd /content
python subscriber.py --out rekaman_terminal.csv
```

Tekan **Ctrl+C** untuk berhenti. Pengaturan bisa diubah lewat opsi, misalnya `--topic`, `--broker`, `--port`, atau `--print-every 50` agar hanya setiap pesan ke-50 yang ditampilkan. Lihat semua opsi dengan `python subscriber.py --help`.

Jika terminal tidak tersedia, perintah yang sama bisa dijalankan dari sel dengan tanda seru di depan: `!python subscriber.py --out rekaman_terminal.csv` (hentikan dengan tombol ■).